In [ ]:
import pandas as pd
import torch
import gc
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split

# 1. Global Config & Setup
MODEL_NAME = "roberta-base"
MAX_LENGTH = 64
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"RUNNING IN {DEVICE}")

variations = {
 "var_1_all_preprocessing":"/content/variation1.csv",
"var_2_normalising_casing_space":"/content/variation2.csv",
"var_3_all_no_emoji_handling":"/content/variation3.csv",
"var_4_all_no_hash_no_elong":"/content/variation4.csv",
"var_5_all_lowercased":"/content/variation5.csv",
}

label_map = {"negative": 0, "neutral": 1, "positive": 2}

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. The Training Loop
for var_name, train_path in variations.items():
  print(f"\n" + "="*50)
  print(f"🏋️‍♂️ TRAINING MODEL: {var_name}")
  print("="*50)

  # Load and map training data
  train_df = pd.read_csv(train_path)

  # Split for Early Stopping (Validation Set)
  train_texts, val_texts, train_labels, val_labels = train_test_split(
  train_df['comment'].tolist(), train_df['sentiment'].tolist(),
  test_size=0.1, random_state=42, stratify=train_df['sentiment']
  )

  # Tokenize
  train_enc = tokenizer(train_texts, padding=True, truncation=True, max_length=MAX_LENGTH)
  val_enc = tokenizer(val_texts, padding=True, truncation=True, max_length=MAX_LENGTH)

  train_ds = SentimentDataset(train_enc, train_labels)
  val_ds = SentimentDataset(val_enc, val_labels)

  # Initialize fresh model
  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)

  training_args = TrainingArguments(
  output_dir=f'./results/results_{var_name}',
  num_train_epochs=5,
  per_device_train_batch_size=BATCH_SIZE,
  eval_strategy="epoch",
  save_strategy="epoch",
  load_best_model_at_end=True,
  metric_for_best_model="eval_loss",
  report_to="none"
  )

  trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=train_ds,
  eval_dataset=val_ds,
  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
  )

  # Train and Save
  trainer.train()
  trainer.save_model(f"./models/saved_model_{var_name}")
  print(f"✅ Model saved successfully to ./saved_model_{var_name}")

  # Clear Memory before the next variation starts
  del model, trainer
  torch.cuda.empty_cache()
  gc.collect()


print("\n🎉 ALL TRAINING COMPLETE!")

RUNNING IN cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved successfully to ./saved_model_var_5_all_lowercased

🎉 ALL TRAINING COMPLETE!


In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Ensure we have the same setup
MODEL_NAME = "roberta-base"
MAX_LENGTH = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

test_variations = {
 "var_1_all_preprocessing":"/content/TEST_DATASET/variation1_all_preprocessing.csv",
"var_2_normalising_casing_space":"/content/TEST_DATASET/variation2_normalising_casing_spaces.csv",
"var_3_all_no_emoji_handling":"/content/TEST_DATASET/variation3_all_preprocessing_except_no_emoji_handling.csv",
"var_4_all_no_hash_no_elong":"/content/TEST_DATASET/variation4_all_preprocessing_except_elongation_hashtag.csv",
"var_5_all_lowercased":"/content/TEST_DATASET/variation5_all_preprocessing_and_lowercased.csv"
}

label_map = {"negative": 0, "neutral": 1, "positive": 2}
inv_map = {v: k for k, v in label_map.items()}

# Redefine metric function for the Trainer
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
all_performance_metrics = []

# 1. The Evaluation Loop
for var_name, test_path in test_variations.items():
    print(f"\n🔍 EVALUATING: {var_name}")

    # Load test data
    maze_df = pd.read_csv(test_path)

    # Tokenize and create dataset
    maze_enc = tokenizer(maze_df['CommentText'].tolist(), padding=True, truncation=True, max_length=MAX_LENGTH)
    maze_ds = SentimentDataset(maze_enc, maze_df['Sentiment'].tolist()) # Reusing the class from Cell 1

    # Load the trained model from disk!
    model_path = f"./models/saved_model_{var_name}"
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(DEVICE)

    # Initialize a dummy trainer just for prediction
    eval_trainer = Trainer(
        model=model,
        compute_metrics=compute_metrics
    )

    # Evaluate Metrics
    eval_results = eval_trainer.evaluate(maze_ds)
    all_performance_metrics.append({
        "Variation": var_name,
        "Accuracy": eval_results['eval_accuracy'],
        "F1_Score": eval_results['eval_f1']
    })

    # Log Misclassifications
    preds_output = eval_trainer.predict(maze_ds)
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    y_true = preds_output.label_ids

    errors = []
    for i in range(len(y_true)):
        if y_pred[i] != y_true[i]:
            errors.append({
                "comment_index": i,
                "comment": maze_df.iloc[i]['CommentText'],
                "true_label": inv_map[y_true[i]],
                "predicted_label": inv_map[y_pred[i]]
            })

    # Save Error CSV
    error_df = pd.DataFrame(errors)
    error_df.to_csv(f"errors_{var_name}.csv", index=False)
    print(f"✅ Metrics recorded and {len(errors)} errors logged for {var_name}")

    # Clean Memory
    del model, eval_trainer
    torch.cuda.empty_cache()

# 2. Final Report Output
metrics_df = pd.DataFrame(all_performance_metrics)
metrics_df.to_csv("final_fine_tuning_ablation_report.csv", index=False)

print("\n🏆 FINAL FINE-TUNING LEADERBOARD:")
display(metrics_df.sort_values(by="Accuracy", ascending=False))


🔍 EVALUATING: var_1_all_preprocessing


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Metrics recorded and 6 errors logged for var_1_all_preprocessing

🔍 EVALUATING: var_2_normalising_casing_space


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Metrics recorded and 15 errors logged for var_2_normalising_casing_space

🔍 EVALUATING: var_3_all_no_emoji_handling


...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Metrics recorded and 6 errors logged for var_4_all_no_hash_no_elong

🔍 EVALUATING: var_5_all_lowercased


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Metrics recorded and 6 errors logged for var_5_all_lowercased

🏆 FINAL FINE-TUNING LEADERBOARD:


,Variation,Accuracy,F1_Score
0,var_1_all_preprocessing,0.976190,0.975990
4,var_5_all_lowercased,0.976190,0.976186
3,var_4_all_no_hash_no_elong,0.976190,0.976196
2,var_3_all_no_emoji_handling,0.964286,0.964152
1,var_2_normalising_casing_space,0.940476,0.940267


In [ ]:
import shutil

# 1. Name of the output zip file (do not include .zip extension)
output_filename = 'models'

# 2. Path to the folder you want to zip
folder_to_zip = '/content/models'

# 3. Create the zip archive
shutil.make_archive(output_filename, 'zip', folder_to_zip)


'/content/models.zip'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# List of file paths (replace with your actual paths)
file_paths = [
    r"/kaggle/working/errors_var_1_all_preprocessing_thresholded.csv",
    r"/kaggle/working/errors_var_2_normalising_casing_space_thresholded.csv",
    r"/kaggle/working/errors_var_3_all_no_emoji_handling_thresholded.csv",
    r"/kaggle/working/errors_var_4_all_no_hash_no_elong_thresholded.csv",
    r"/kaggle/working/errors_var_5_all_lowercased_thresholded.csv"
]

# Step 1: Store comment indices in 5 separate sets
comment_index_sets = []

for path in file_paths:
    df = pd.read_csv(path)
    index_set = set(df["comment_index"].tolist())
    comment_index_sets.append(index_set)

# Now you have 5 sets
set1, set2, set3, set4, set5 = comment_index_sets

# Step 2: Combine all misclassified data
all_data = pd.concat([pd.read_csv(path) for path in file_paths], ignore_index=True)

# Step 3: Get unique labels
labels = sorted(list(set(all_data["true_label"]).union(set(all_data["predicted_label"]))))

# Step 4: Initialize confusion matrix
label_to_idx = {label: i for i, label in enumerate(labels)}
n = len(labels)
conf_matrix = np.zeros((n, n), dtype=int)

# Step 5: Populate confusion matrix (only misclassifications)
for _, row in all_data.iterrows():
    true = row["true_label"]
    pred = row["predicted_label"]
    
    if true != pred:  # ensure misclassified only
        i = label_to_idx[true]
        j = label_to_idx[pred]
        conf_matrix[i][j] += 1

# Step 6: Print matrix
print("Labels:", labels)
print("\nConfusion Matrix (Misclassified Only):\n")
print(conf_matrix)

# Step 7: Plot heatmap (optional but useful)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Misclassified Samples Only)")
plt.show()